In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Machine Learning
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
#  DATA ACQUISITION & EXPLORATION 
print("--- Loading Data ---")
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['Price'] = data.target

print(f"Dataset Shape: {df.shape}")
print(df.head())
# Visualizing Correlations
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
# DATA PREPROCESSING 
X = df.drop('Price', axis=1)
y = df['Price']

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Scaling 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# BASELINE MODEL: LINEAR REGRESSION
print("\nTraining Baseline: Linear Regression ")
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_test_scaled)
print(f"LR R2 Score: {r2_score(y_test, lr_preds):.4f}")
print(f"LR MAE: {mean_absolute_error(y_test, lr_preds):.4f}")

In [ ]:
# IMPROVED MODEL: RANDOM FOREST 
print("\nTraining Improved Model: Random Forest ")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train) # Tree-based models don't strictly require scaling

In [ ]:
rf_preds = rf_model.predict(X_test)
print(f"RF R2 Score: {r2_score(y_test, rf_preds):.4f}")
print(f"RF MAE: {mean_absolute_error(y_test, rf_preds):.4f}")

In [ ]:
# ADVANCED MODEL: DEEP NEURAL NETWORK 
print("\nTraining Deep Neural Network ")

# Define a robust architecture
dnn_model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1) # Output layer for regression
])

In [ ]:
dnn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
# Training
history = dnn_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=0
)

In [ ]:
# Evaluate DNN
dnn_preds = dnn_model.predict(X_test_scaled).flatten()
print(f"DNN R2 Score: {r2_score(y_test, dnn_preds):.4f}")
print(f"DNN MAE: {mean_absolute_error(y_test, dnn_preds):.4f}")

In [ ]:
#  TRANSITION TO CNN 
# To use a CNN on tabular data, we reshape the 8 features into a "pseudo-image" (2x4)
print("\nTransitioning to CNN Architecture ")

X_train_cnn = X_train_scaled.reshape(-1, 2, 4, 1)
X_test_cnn = X_test_scaled.reshape(-1, 2, 4, 1)

In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (2, 2), activation='relu', padding='same', input_shape=(2, 4, 1)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

In [ ]:
cnn_model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
cnn_model.fit(X_train_cnn, y_train, epochs=30, batch_size=32, verbose=0)

In [ ]:
cnn_preds = cnn_model.predict(X_test_cnn).flatten()
print(f"CNN R2 Score: {r2_score(y_test, cnn_preds):.4f}")

In [ ]:
#  FINAL COMPARISON VISUALIZATION 
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Deep Neural Network', 'CNN'],
    'R2 Score': [
        r2_score(y_test, lr_preds),
        r2_score(y_test, rf_preds),
        r2_score(y_test, dnn_preds),
        r2_score(y_test, cnn_preds)
    ]
})
plt.figure(figsize=(10, 6))
sns.barplot(x='R2 Score', y='Model', data=results.sort_values(by='R2 Score', ascending=False))
plt.title("Model Performance Comparison (R2 Score)")
plt.xlim(0, 1)
plt.show()

print("\nPipeline Complete. Best model identified.")

# Second Approach

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

In [ ]:
# Scikit-Learn Utilities
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    r2_score, 
    explained_variance_score
)

In [ ]:
# Classic ML Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

In [ ]:
# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

In [ ]:
# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
class DataEngine:
    """Handles data acquisition, cleaning, and feature engineering."""
    
    def __init__(self):
        self.data = None
        self.X = None
        self.y = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.scaler = StandardScaler()
    def load_and_prepare(self):
        logger.info("Fetching California Housing Dataset...")
        dataset = fetch_california_housing()
        self.data = pd.DataFrame(dataset.data, columns=dataset.feature_names)
        self.data['Target'] = dataset.target

        # Feature Engineering: Adding household density
        self.data['AveOccupancyPerRoom'] = self.data['AveOccup'] / self.data['AveRooms']
        self.X = self.data.drop('Target', axis=1)
        self.y = self.data['Target']
        
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=42
        )
        logger.info(f"Data split successful. Training shape: {self.X_train.shape}")
        return self.X_train, self.X_test, self.y_train, self.y_test

    def perform_eda(self):
        logger.info("Generating EDA Visuals...")
        plt.figure(figsize=(12, 8))
        sns.heatmap(self.data.corr(), annot=True, cmap='viridis', fmt='.2f')
        plt.title("Feature Correlation Matrix")
        plt.show()

        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        sns.histplot(self.y, kde=True, ax=axes[0], color='blue').set_title("Target Distribution")
        sns.scatterplot(data=self.data, x='Longtitude', y='Latitude', hue='Target', ax=axes[1], palette='magma')
        axes[1].set_title("Geographical Distribution vs Price")
        plt.show()


In [ ]:
class ModelEvolution:
    """Systematically builds and evaluates models from baseline to DL."""
    
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.results = {}
        # Scaling for distance-based/gradient models
        self.scaler = StandardScaler()
        self.X_train_scaled = self.scaler.fit_transform(X_train)
        self.X_test_scaled = self.scaler.transform(X_test)
    
    def run_baseline_comparison(self):
        """Phase 1: Linear Models and Ensembles."""
        models_to_test = {
            "LinearRegression": LinearRegression(),
            "Ridge": Ridge(alpha=1.0),
            "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
            "GradientBoosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1)
        }
        for name, model in models_to_test.items():
            logger.info(f"Training {name}...")
            # Use scaled data for LR/Ridge, raw for Trees (though scaled works too)
            train_data = self.X_train_scaled if name in ["LinearRegression", "Ridge"] else self.X_train
            test_data = self.X_test_scaled if name in ["LinearRegression", "Ridge"] else self.X_test
            
            model.fit(train_data, self.y_train)
            preds = model.predict(test_data)